# 02 · 编码器模块（Encoders）功能演示

演示 9 种特征编码器、双 API 风格、未知/缺失值处理与编码映射的导出导入。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. 九种编码器总览
WOE / Target / Count / OneHot / Ordinal / Quantile / CatBoost / GBM / Cardinality，均遵循 sklearn Transformer 接口。

In [2]:
from hscredit.core.encoders import (WOEEncoder, TargetEncoder, CountEncoder, OneHotEncoder,
    OrdinalEncoder, QuantileEncoder, CatBoostEncoder, GBMEncoder, CardinalityEncoder)

df2 = df.copy()
df2['城市'] = (df2['客户编号'] % 30).astype(str) + '城'   # 构造高基数类别特征
Xc = df2[[CAT_FEATURE, '城市']].astype(str)

encoders = {
    'WOE': WOEEncoder(), 'Target': TargetEncoder(), 'Count': CountEncoder(),
    'OneHot': OneHotEncoder(), 'Ordinal': OrdinalEncoder(), 'Quantile': QuantileEncoder(),
    'CatBoost': CatBoostEncoder(), 'GBM': GBMEncoder(), 'Cardinality': CardinalityEncoder(),
}
shapes = []
for name, enc in encoders.items():
    out = enc.fit_transform(Xc, y)
    shapes.append({'编码器': name, '输出列数': out.shape[1]})
pd.DataFrame(shapes)

,编码器,输出列数
0,WOE,2
1,Target,2
2,Count,2
3,OneHot,21
4,Ordinal,2
5,Quantile,2
6,CatBoost,2
7,GBM,100
8,Cardinality,2


## 2. WOE 编码：映射与 IV（带正则化平滑，避免 0 坏样本类别 WOE 发散）

In [3]:
woe_enc = WOEEncoder()
woe_enc.fit(Xc[[CAT_FEATURE]], y)
mapping = woe_enc.get_mapping(CAT_FEATURE)
pd.DataFrame({'类别': list(mapping.keys()), 'WOE': [round(v,4) for v in mapping.values()]})

,类别,WOE
0,礼包,-0.1386
1,珠宝首饰,0.1242
2,手机通讯,-0.2702
3,智能设备,0.1919
4,电脑数码,-0.0445
5,家用电器,0.7028
6,NaN,0.0000
7,__UNKNOWN__,0.0000


## 3. 双 API 风格 + 未知类别 / 缺失值处理

In [4]:
# scorecardpipeline 风格
df_t = Xc.copy(); df_t['target'] = y.values
woe_scp = WOEEncoder(target='target').fit_transform(df_t)
# 未知类别在 transform 时按 handle_unknown 策略处理
Xnew = Xc.copy(); Xnew.iloc[0, 0] = '__未见过的类别__'
out_unknown = woe_enc.transform(Xnew[[CAT_FEATURE]])
print('未知类别编码值:', round(float(out_unknown.iloc[0, 0]), 4))
woe_scp.head(3)

未知类别编码值: 0.0


,商品类别,城市,target
0,-0.1386,0.2555,0
1,-0.1386,0.2555,0
2,0.1242,-0.3581,0


## 4. 编码映射导出 / 导入（部署一致性）

In [5]:
mp = woe_enc.export_mapping()
woe_enc2 = WOEEncoder(); woe_enc2.import_mapping(mp)
same = woe_enc.transform(Xc[[CAT_FEATURE]]).round(6).equals(woe_enc2.transform(Xc[[CAT_FEATURE]]).round(6))
print('导出/导入后编码一致:', same)

导出/导入后编码一致: True


## 5. 编码结果导出到 Excel

In [6]:
res = Xc[[CAT_FEATURE]].copy()
res['WOE编码'] = woe_enc.transform(Xc[[CAT_FEATURE]])[CAT_FEATURE].values
res['Target编码'] = TargetEncoder().fit_transform(Xc[[CAT_FEATURE]], y)[CAT_FEATURE].values
res.drop_duplicates().to_excel(f"{OUT}/02_encoders_mapping.xlsx", index=False)
print('已保存编码对照表')
res.drop_duplicates().reset_index(drop=True)

已保存编码对照表


,商品类别,WOE编码,Target编码
0,礼包,-0.1386,0.1218
1,珠宝首饰,0.1242,0.1563
2,手机通讯,-0.2702,0.1066
3,智能设备,0.1919,0.0280
4,电脑数码,-0.0445,0.1019
5,家用电器,0.7028,0.0467
